# Tokens & Costs
### LLM 101 — Module 0 — Foundations

> Every LLM API call is a **metered transaction**. This notebook teaches you the meter.

---

## 📺 Watch the episode alongside this notebook

Run the cells as you watch. Nothing here is a spectator sport — every concept has a demo you can break.

## What you'll be able to do by the end

1. Explain what a token *actually* is, and why `"hello"` and `" hello"` are different tokens
2. Predict which of two prompts costs more — before sending either
3. Count tokens correctly for **OpenAI** (`tiktoken`) and **Claude** (`count_tokens` API), and know why mixing them up burns money
4. Price any request in dollars, and project a whole product's monthly bill
5. Spot the **O(n²) conversation-history trap** that silently 10×'s a chat app's bill
6. Use prompt caching, and prove it's working
7. Ship a `CostTracker` with a hard budget ceiling — the thing you actually put in production

---

## ⚙️ Setup

```bash
source .venv/bin/activate     # Mac/Linux  (.venv\Scripts\activate on Windows)
pip install -r requirements.txt
cp .env.example .env          # then add your keys
jupyter lab
```

### 🔑 Do I need API keys?

**Most of this notebook runs with zero API keys and zero dollars spent.** Anything that
calls a real API is clearly marked **`💸 LIVE API`** and is skipped automatically if the
key is missing.

| Part | What it covers | Needs a key? |
|------|----------------|--------------|
| 1 · Anatomy of a token | tokenizers, boundaries, encodings | ❌ free |
| 2 · Token density | why JSON and Hindi cost more than English | ❌ free |
| 3 · The envelope tax | your count vs the API's count | ⚠️ partly |
| 4 · Pricing model | dollars per request | ❌ free |
| 5 · Scaling to a product | the O(n²) trap | ❌ free |
| 6 · Prompt caching | 90% discounts, and how to break them | ⚠️ partly |
| 7 · Budget guardrails | `CostTracker`, preflight checks | ❌ free |
| 8 · Context windows | the 1M-context trap | ❌ free |

> 💡 **One-time network note:** `tiktoken` downloads its vocabulary file on first use
> (a few MB, cached in `~/.cache/tiktoken` afterwards). If the first cell hangs or errors,
> that download is what's failing — check your connection or proxy.

---

## 🧠 Frontend brain, meet the LLM

You already have the mental model for this. You just call it something else:

| You know this | Same idea here |
|---|---|
| Bundle size in KB | Prompt size in tokens |
| `Content-Length` on a request | `usage.input_tokens` |
| An n+1 query blowing up your API bill | Resending chat history every turn |
| A CDN cache hit | Prompt cache hit (~90% cheaper) |
| Tree-shaking dead code out of a bundle | Trimming dead context out of a prompt |

The difference: your bundle ships once and gets cached by the browser. **Your prompt ships
on every single keystroke-triggered request, and you pay for it every time.**

---

## 0 · Setup & capability check

Run this first. It tells you exactly which demos below will run on your machine.

In [ ]:
import os
import sys
import json
from dataclasses import dataclass, field

import matplotlib.pyplot as plt
import tiktoken
from dotenv import load_dotenv

load_dotenv()  # reads .env from the repo root

OPENAI_KEY = os.getenv("OPENAI_API_KEY", "").strip()
ANTHROPIC_KEY = os.getenv("ANTHROPIC_API_KEY", "").strip()

# A key that's still the placeholder from .env.example doesn't count.
HAS_OPENAI = OPENAI_KEY.startswith("sk-") and "your-key-here" not in OPENAI_KEY
HAS_ANTHROPIC = ANTHROPIC_KEY.startswith("sk-ant-")

_ENC_CACHE = {}


def get_encoding(name: str = "o200k_base"):
    """Load a tiktoken encoding, with a human-readable error if the download fails."""
    if name not in _ENC_CACHE:
        try:
            _ENC_CACHE[name] = tiktoken.get_encoding(name)
        except Exception as exc:  # network / proxy / disk-cache problems
            raise RuntimeError(
                f"Could not load the '{name}' tokenizer.\n"
                "tiktoken downloads its vocabulary on first use and caches it in "
                "~/.cache/tiktoken.\n"
                "Fix: check your internet connection or corporate proxy, then re-run.\n"
                f"Original error: {exc}"
            ) from exc
    return _ENC_CACHE[name]


enc = get_encoding("o200k_base")  # the encoding used by GPT-4o and newer

print(f"Python           {sys.version.split()[0]}")
print(f"tiktoken         {tiktoken.__version__}")
print(f"tokenizer ready  o200k_base  ({enc.n_vocab:,} tokens in vocabulary)")
print()
print(f"{'OPENAI_API_KEY':<18} {'✅ found' if HAS_OPENAI else '⬜ not set — live OpenAI demos will be skipped'}")
print(f"{'ANTHROPIC_API_KEY':<18} {'✅ found' if HAS_ANTHROPIC else '⬜ not set — live Claude demos will be skipped'}")
print()
print("Everything except the 💸 LIVE API cells runs fine either way. Let's go.")

---
---

# Part 1 · Anatomy of a token

## The one-sentence version

> An LLM cannot read text. It reads **integers**. A tokenizer is the codec that turns your
> string into a list of integers, and back again.

That's it. `enc.encode(text) → [int]` and `enc.decode([int]) → text`. Everything else in
this part is you developing an intuition for *where the boundaries land*.

### The frontend analogy

It's `TextEncoder`, but the alphabet was learned from data instead of defined by a spec:

```js
// JavaScript: a fixed, spec-defined codec
new TextEncoder().encode("hello")   // Uint8Array [104, 101, 108, 108, 111]  — 5 bytes

// LLM tokenizer: a *learned* codec, where common chunks got their own ID
enc.encode("hello")                 // [15339]                               — 1 token
```

The tokenizer was trained on a giant corpus with an algorithm called **BPE** (Byte-Pair
Encoding). It started with single bytes and repeatedly merged the most frequent adjacent
pair into a new symbol. So `" the"` — wildly common — earned a single ID. `" thé"` did not.

**Common text compresses. Rare text does not.** That single sentence explains 90% of the
weird cost behaviour you'll see for the rest of your career.

### Demo 1.1 — See the boundaries

Let's stop talking and look at one.

In [ ]:
def show_tokens(text: str, encoding_name: str = "o200k_base", label: str | None = None):
    """Print a string split into its actual tokens, with IDs."""
    e = get_encoding(encoding_name)
    ids = e.encode(text)
    # decode_single_token_bytes gives the exact bytes of one token
    pieces = [e.decode_single_token_bytes(i).decode("utf-8", errors="replace") for i in ids]

    header = label or f"{encoding_name}"
    print(f"── {header} " + "─" * max(0, 62 - len(header)))
    print(f'   text   : {text!r}')
    print(f"   chars  : {len(text)}")
    print(f"   tokens : {len(ids)}")
    # '·' marks a leading space so you can SEE that spaces belong to the token
    print("   split  : " + " │ ".join(p.replace(" ", "·") for p in pieces))
    print(f"   ids    : {ids}")
    print()
    return ids


show_tokens("Hello world!")
show_tokens("The quick brown fox jumps over the lazy dog.")
show_tokens("const [count, setCount] = useState(0);")

Look closely at that third one.

`useState` is one of the most-typed words on the internet, so BPE may well have given it
its own token. But `setCount` — a name *you* invented — gets shredded into pieces.

**That's the whole game:** the tokenizer rewards you for being ordinary and taxes you for
being novel.

Also notice the `·` markers: **the space belongs to the token that follows it.** `" world"`
is one token, not a space plus `"world"`. This trips up everyone once.

### Demo 1.2 — Four ways to accidentally double your token count

Same word, four spellings. Watch the counts.

In [ ]:
variants = [
    ("hello",       "plain, no leading space"),
    (" hello",      "leading space — the common case mid-sentence"),
    ("Hello",       "capitalised"),
    ("HELLO",       "shouting"),
    ("hello!",      "with punctuation"),
    ("helo",        "typo"),
    ("h e l l o",   "spaced out"),
    ("hellohello",  "doubled, no separator"),
]

print(f"{'string':<16} {'tokens':>7}  {'ids':<26} note")
print("─" * 84)
for s, note in variants:
    ids = enc.encode(s)
    print(f"{s!r:<16} {len(ids):>7}  {str(ids):<26} {note}")

Read that table twice. Things worth internalising:

- **`"hello"` and `" hello"` are different tokens with different IDs.** The model has learned
  two separate embeddings for them.
- **Casing costs money.** `HELLO` is usually several tokens because SCREAMING is rarer than
  lowercase.
- **A typo is expensive.** `helo` isn't in the vocabulary as a unit, so it gets decomposed.
  Same reason: user-generated text, IDs, and hashes cost far more than clean prose.
- **Spacing out a word multiplies its cost.** Relevant if you're ever tempted to "format"
  a prompt for readability.

> 🎯 **Practical takeaway:** if you're near a context or budget limit, normalising
> user input (trimming, collapsing whitespace) is free money.

### Demo 1.3 — Three different meanings of "length"

Here's a bug I want you to never write.

In JavaScript, `"length"` already lies to you — `.length` counts UTF-16 code units, not
characters, which is why emoji break your character counters:

```js
"café".length        // 4
"🚀".length          // 2   ← one emoji, two code units
"👨‍👩‍👧".length      // 8   ← one family, eight code units
```

Tokens are a **third, completely unrelated** unit. If you're validating "max 500 characters"
on the frontend and the backend is enforcing a token budget, those two numbers have nothing
to do with each other.

In [ ]:
def utf16_units(s: str) -> int:
    """What JavaScript's .length would return for this string."""
    return len(s.encode("utf-16-le")) // 2


samples = [
    "café",
    "🚀",
    "👨‍👩‍👧",
    "hello world",
    "नमस्ते दुनिया",          # Hindi: "hello world"
    "こんにちは世界",           # Japanese
    "a1b2c3d4-e5f6-7890",   # id fragment
]

print(f"{'string':<22} {'py len':>7} {'JS .length':>11} {'utf-8 bytes':>12} {'tokens':>7}")
print("─" * 64)
for s in samples:
    print(
        f"{s:<22} {len(s):>7} {utf16_units(s):>11} "
        f"{len(s.encode('utf-8')):>12} {len(enc.encode(s)):>7}"
    )

Four columns, four different numbers, for the same string.

**The only column that costs you money is the last one.** Build your budgets, your
truncation, and your `maxLength` validators on that column — not on `.length`.

Look at the Hindi row especially. Same *meaning* as "hello world", but a multiple of the
tokens. We'll dig into why in Part 2, and it has real product consequences: **your
non-English users are more expensive to serve than your English users.**

### Demo 1.4 — Different models, different tokenizers

`cl100k_base` powers GPT-3.5 / GPT-4 / GPT-4-turbo and the OpenAI embedding models.
`o200k_base` powers GPT-4o and later. Bigger vocabulary → fewer tokens for the same text.

A token count is only meaningful **relative to a specific model.**

In [ ]:
probe = {
    "English prose":  "The rapid advancement of artificial intelligence has transformed software.",
    "TypeScript":     "export const useUser = (id: string): User | null => cache.get(id) ?? null;",
    "JSON":           '{"userId": 12345, "isActive": true, "roles": ["admin", "editor"]}',
    "Hindi":          "कृत्रिम बुद्धिमत्ता ने सॉफ़्टवेयर को बदल दिया है।",
    "Emoji":          "🚀🔥💡✨🎯",
}

cl100k = get_encoding("cl100k_base")
o200k = get_encoding("o200k_base")

print(f"{'content':<16} {'cl100k':>8} {'o200k':>8} {'change':>9}")
print("─" * 45)
for name, text in probe.items():
    a, b = len(cl100k.encode(text)), len(o200k.encode(text))
    delta = (b - a) / a * 100
    print(f"{name:<16} {a:>8} {b:>8} {delta:>8.0f}%")

> ⚠️ **The trap this sets up:** these are both *OpenAI* tokenizers. **Neither one is
> correct for Claude, Gemini, Llama, or Mistral.** Every model family ships its own
> tokenizer, and using the wrong one produces confidently wrong numbers.
>
> We'll measure exactly how wrong in Part 3. It's worse than you'd guess.

---
---

# Part 2 · Token density — why some text costs 5× more

You've probably heard the rule of thumb:

> "1 token ≈ 4 characters ≈ ¾ of a word"

It's true **for English prose** and dangerously wrong for everything else. Let's measure
the real spread. The metric we care about is **characters per token** — higher is cheaper.

### Demo 2.1 — A density benchmark across content types

In [ ]:
CORPUS = {
    "English prose": (
        "The team shipped the new dashboard on Friday afternoon. "
        "Users reported that the page loaded noticeably faster than before."
    ),
    "Technical prose": (
        "Memoize the selector to avoid recomputing derived state on every render, "
        "then hydrate the cache from the server payload during initial mount."
    ),
    "TypeScript code": (
        "export async function fetchUser(id: string): Promise<User | null> {\n"
        "  const res = await fetch(`/api/users/${id}`);\n"
        "  if (!res.ok) return null;\n"
        "  return (await res.json()) as User;\n"
        "}"
    ),
    "JSON (pretty)": json.dumps(
        {"userId": 84213, "email": "dev@example.com", "isActive": True,
         "roles": ["admin", "editor"], "lastSeenAt": "2026-03-14T09:21:00Z"},
        indent=2,
    ),
    "Minified CSS": ".btn{display:flex;align-items:center;gap:.5rem;padding:8px 16px}",
    "UUIDs": "b3f1c2d4-5e6a-4f70-8b91-0a2c3d4e5f60, 7a1e9c88-2b3d-4e5f-a061-9c8d7e6f5a4b",
    "Base64 blob": "iVBORw0KGgoAAAANSUhEUgAAAAEAAAABCAYAAAAfFcSJAAAADUlEQVR42mP8z8BQDwAEhQGAhKmMIQAAAABJRU5ErkJggg==",
    "Hindi": "टीम ने शुक्रवार दोपहर को नया डैशबोर्ड लॉन्च किया। उपयोगकर्ताओं ने बताया कि पेज पहले से तेज़ी से लोड हुआ।",
    "Japanese": "チームは金曜日の午後に新しいダッシュボードをリリースしました。ページの読み込みが以前より速くなったと報告されています。",
    "Emoji": "🚀🔥💡✨🎯🧠📦⚡️🛠️🎨",
}

rows = []
for name, text in CORPUS.items():
    n_tokens = len(enc.encode(text))
    rows.append({
        "content": name,
        "chars": len(text),
        "tokens": n_tokens,
        "chars_per_token": len(text) / n_tokens,
    })

rows.sort(key=lambda r: r["chars_per_token"], reverse=True)

print(f"{'content type':<18} {'chars':>7} {'tokens':>7} {'chars/token':>12}   cheap ←→ expensive")
print("─" * 84)
for r in rows:
    bar = "█" * max(1, round(r["chars_per_token"] * 3))
    print(f"{r['content']:<18} {r['chars']:>7} {r['tokens']:>7} {r['chars_per_token']:>12.2f}   {bar}")

print()
best, worst = rows[0], rows[-1]
print(f"Spread: {best['content']} is {best['chars_per_token'] / worst['chars_per_token']:.1f}× "
      f"more token-efficient than {worst['content']}.")

### Demo 2.2 — Chart it

In [ ]:
labels = [r["content"] for r in rows]
values = [r["chars_per_token"] for r in rows]
colors = ["#22c55e" if v >= 3.5 else "#f59e0b" if v >= 2 else "#ef4444" for v in values]

fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(labels, values, color=colors)
ax.invert_yaxis()
ax.axvline(4.0, ls="--", lw=1, color="#64748b")
ax.text(4.05, len(labels) - 0.4, ' the "4 chars ≈ 1 token" rule', fontsize=9, color="#64748b")
ax.set_xlabel("characters per token  (higher = cheaper)")
ax.set_title("Token density is not a constant", fontsize=13, weight="bold")
for i, v in enumerate(values):
    ax.text(v + 0.08, i, f"{v:.2f}", va="center", fontsize=9)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

### What the chart is telling you

- **English prose sits near the rule of thumb.** Everything the rule was derived from.
- **Code is denser than you'd hope.** Punctuation, camelCase, and your invented identifiers
  all fragment. A 500-line file is *not* 500 × 4 characters worth of tokens.
- **UUIDs and base64 are token bombs.** They're near-random, so BPE can't compress them at
  all — roughly 1 token per 1–2 characters. Never paste a base64 image into a prompt "just
  to see". Never dump raw IDs when a short label would do.
- **Non-Latin scripts cost multiples.** This is a real, documented fairness problem in LLM
  pricing, and if you serve a global product it's a real line item.

> 🎯 **Product decision this drives:** if you're building for India, Japan, or the Middle East,
> your per-user cost model is not the one from the English-language blog post you read.
> Measure it with *your* users' actual text.

### Demo 2.3 — The same data, four ways

This is the most immediately useful demo in Part 2.

You have a list of users to hand to the model. As a frontend dev your instinct is
`JSON.stringify(data, null, 2)` — it's readable, it's what your API returns. But you're
paying for every space and every repeated key.

**Watch what happens when you stop sending the keys 50 times.**

In [ ]:
users = [
    {"id": i, "name": n, "role": r, "active": a}
    for i, (n, r, a) in enumerate(
        [("Ada Lovelace", "admin", True), ("Grace Hopper", "editor", True),
         ("Alan Turing", "viewer", False), ("Katherine Johnson", "admin", True),
         ("Linus Torvalds", "editor", False)] * 10,
        start=1,
    )
]


def as_pretty_json(rows):
    return json.dumps(rows, indent=2)


def as_compact_json(rows):
    return json.dumps(rows, separators=(",", ":"))


def as_csv(rows):
    head = ",".join(rows[0].keys())
    body = "\n".join(",".join(str(v) for v in r.values()) for r in rows)
    return f"{head}\n{body}"


def as_markdown_table(rows):
    head = "| " + " | ".join(rows[0].keys()) + " |"
    sep = "|" + "|".join(["---"] * len(rows[0])) + "|"
    body = "\n".join("| " + " | ".join(str(v) for v in r.values()) + " |" for r in rows)
    return f"{head}\n{sep}\n{body}"


formats = {
    "JSON (indent=2)": as_pretty_json(users),
    "JSON (minified)": as_compact_json(users),
    "Markdown table": as_markdown_table(users),
    "CSV": as_csv(users),
}

baseline = len(enc.encode(formats["JSON (indent=2)"]))

print(f"50 user records, encoded four ways\n")
print(f"{'format':<20} {'chars':>7} {'tokens':>7} {'vs pretty JSON':>16}")
print("─" * 54)
for name, text in formats.items():
    t = len(enc.encode(text))
    saving = (baseline - t) / baseline * 100
    flag = "  ←  baseline" if name == "JSON (indent=2)" else f"  {saving:+.0f}%"
    print(f"{name:<20} {len(text):>7} {t:>7} {t/baseline:>15.2f}×{flag}")

print("\n─── what CSV actually looks like to the model ───")
print(as_csv(users[:3]))

### Read the savings as money

That's typically **50–70% fewer tokens for identical information.** On a RAG pipeline that
stuffs 50 rows into every single request, that's not a micro-optimisation — it's most of
your bill.

**When to use which:**

| Format | Use it when |
|---|---|
| CSV | Flat, uniform rows. Cheapest by a mile. |
| Markdown table | Flat rows *and* you want the model to reason about them — models read these very well |
| Minified JSON | Nested / ragged data where structure genuinely matters |
| Pretty JSON | Basically never, in a prompt. Indentation is pure cost. |

> ⚠️ **One caveat, and it matters:** cheapest ≠ best. If dropping to CSV makes the model's
> answers worse, you've saved money on a worse product. The honest workflow is: switch
> format, then re-run your evals (Module 6). Measure both axes.

---
---

# Part 3 · The envelope tax — your count vs the API's count

Here's where beginners' estimates start drifting from their invoices.

You don't send a *string* to a chat API. You send a **list of messages**, each with a role.
The API wraps every message in special formatting tokens before the model ever sees it —
role markers, message delimiters, and a primer that tells the model "your turn now".

You pay for those too.

### The frontend analogy

You're not paying for your JSON payload. You're paying for the payload **plus the HTTP
headers**. Small on one big request, brutal on a thousand small ones.

### Demo 3.1 — Naive count vs chat-format count

This is the counting function from OpenAI's own cookbook. Read it — it's short, and the
constants are the whole lesson.

In [ ]:
def num_tokens_from_messages(messages, encoding_name: str = "o200k_base") -> int:
    """Token count for a chat-completions request (OpenAI chat format).

    Each message is wrapped as roughly:
        <|start|>{role}<|message|>{content}<|end|>
    ...which is the 3 tokens of overhead per message. Then the whole request is
    primed with <|start|>assistant<|message|> so the model knows to reply: +3.
    """
    e = get_encoding(encoding_name)
    tokens_per_message = 3
    tokens_per_name = 1

    total = 0
    for message in messages:
        total += tokens_per_message
        for key, value in message.items():
            total += len(e.encode(str(value)))
            if key == "name":
                total += tokens_per_name
    total += 3  # priming the assistant reply
    return total


messages = [
    {"role": "system", "content": "You are a terse assistant. Answer in one sentence."},
    {"role": "user", "content": "What is a React key prop for?"},
]

# The naive way most people estimate: concatenate and count.
naive = len(enc.encode("".join(m["content"] for m in messages)))
proper = num_tokens_from_messages(messages)

print(f"naive string count      : {naive:>4} tokens")
print(f"actual chat-format count: {proper:>4} tokens")
print(f"envelope overhead       : {proper - naive:>4} tokens  ({(proper-naive)/naive*100:.0f}% more)")

### Why you should care about ~9 tokens

On one request: noise. Now scale it.

The overhead is **per message**, and a chat app resends the *entire history* every turn —
so the envelope tax grows with conversation length, on every single request.

In [ ]:
print(f"{'turns in history':>17} {'messages':>9} {'content tk':>11} {'envelope tk':>12} {'% waste':>8}")
print("─" * 62)

history = [{"role": "system", "content": "You are a helpful assistant."}]
for turn in range(1, 41):
    history.append({"role": "user", "content": "Can you explain that a bit more?"})
    history.append({"role": "assistant", "content": "Sure. Here's a short explanation."})

    if turn in (1, 5, 10, 20, 40):
        content = len(enc.encode("".join(m["content"] for m in history)))
        total = num_tokens_from_messages(history)
        print(f"{turn:>17} {len(history):>9} {content:>11} {total - content:>12} "
              f"{(total - content) / total * 100:>7.0f}%")

print("\nShort messages + long history = the envelope becomes a real share of the bill.")

### Demo 3.2 — 💸 LIVE API: prove it against the real thing

Estimates are worth nothing until you've checked them once. Every provider returns a
`usage` object on the response — **that** is what you're billed for. Treat it as ground truth.

> 💰 **Cost of running this cell: well under one cent.** It's a tiny prompt on a cheap model.

In [ ]:
if not HAS_OPENAI:
    print("⬜ Skipped — set OPENAI_API_KEY in .env to run this cell.")
else:
    from openai import OpenAI

    client = OpenAI()

    demo_messages = [
        {"role": "system", "content": "You are a terse assistant. Answer in one sentence."},
        {"role": "user", "content": "What is a React key prop for?"},
    ]

    estimate = num_tokens_from_messages(demo_messages)

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=demo_messages,
        max_tokens=60,
    )

    usage = response.usage
    print("model reply:", response.choices[0].message.content, "\n")
    print(f"my estimate        : {estimate:>4} input tokens")
    print(f"API says (billed)  : {usage.prompt_tokens:>4} input tokens")
    print(f"difference         : {usage.prompt_tokens - estimate:>+4}")
    print()
    print(f"output tokens      : {usage.completion_tokens:>4}")
    print(f"total tokens       : {usage.total_tokens:>4}")
    print()
    print("👉 If the difference is 0, your estimator is exact for this model.")
    print("   If it's not, trust `usage` — and use it to calibrate your estimator.")

### Demo 3.3 — 💸 LIVE API: the expensive mistake — tiktoken on Claude

This is the single most valuable cell in Part 3.

`tiktoken` is **OpenAI's** tokenizer. Claude has a different one, and Anthropic doesn't
publish it as a local library. If you estimate Claude costs with `tiktoken`, you will
**systematically undercount** — typically 15–20% on English prose, and considerably more on
code and non-English text.

The right tool is Anthropic's `count_tokens` endpoint. It's a real API call, but it's
**free** and it returns the exact number you'll be billed for.

> 💰 **Cost of running this cell: $0.00.** `count_tokens` is not billed.

In [ ]:
if not HAS_ANTHROPIC:
    print("⬜ Skipped — set ANTHROPIC_API_KEY in .env to run this cell.")
    print("   (Worth doing: this is the demo that saves people from a surprise invoice.)")
else:
    import anthropic

    aclient = anthropic.Anthropic()
    CLAUDE_MODEL = "claude-opus-5"

    trials = {
        "English prose": (
            "The team shipped the new dashboard on Friday afternoon. Users reported "
            "that the page loaded noticeably faster than before."
        ),
        "TypeScript code": (
            "export async function fetchUser(id: string): Promise<User | null> {\n"
            "  const res = await fetch(`/api/users/${id}`);\n"
            "  if (!res.ok) return null;\n"
            "  return (await res.json()) as User;\n"
            "}"
        ),
        "Hindi": "टीम ने शुक्रवार दोपहर को नया डैशबोर्ड लॉन्च किया।",
    }

    print(f"{'content':<18} {'tiktoken':>9} {'Claude (real)':>14} {'undercount':>11}")
    print("─" * 56)
    for name, text in trials.items():
        guess = len(enc.encode(text))
        real = aclient.messages.count_tokens(
            model=CLAUDE_MODEL,
            messages=[{"role": "user", "content": text}],
        ).input_tokens
        print(f"{name:<18} {guess:>9} {real:>14} {(real - guess) / real * 100:>10.1f}%")

    print("\n👉 Rule: count with the tokenizer that belongs to the model you're calling.")
    print("   OpenAI  → tiktoken (local, free, instant)")
    print("   Claude  → client.messages.count_tokens(...) (API call, free, exact)")

### 📌 The rule, written down

| Provider | How to count | Cost | Gotcha |
|---|---|---|---|
| OpenAI | `tiktoken` locally | free | pick the right encoding for the model |
| Anthropic | `client.messages.count_tokens(...)` | free API call | **never** use `tiktoken` here |
| Everyone | read `response.usage` after the call | free | this is the billed truth — log it |

**Estimate before you send** (to enforce budgets and catch runaway prompts).
**Record `usage` after you send** (because that's what you actually pay).

Do both. Part 7 wires them together into something you can ship.

---
---

# Part 4 · From tokens to dollars

Now the fun part. Two things you need to internalise about LLM pricing:

**1. It's priced per million tokens.** Which makes every individual call look free, and is
exactly why people get surprised at the end of the month.

**2. Output costs 4–5× more than input.** Generating is expensive; reading is cheap. This
asymmetry should drive your design decisions, and almost nobody's does.

---

### ⚠️ Read this before you trust the numbers below

Model prices change, and new models ship constantly. The table below is a **dated snapshot**,
not gospel.

This is also a genuine engineering lesson: **keep pricing in exactly one place, stamp it with
a date, and put the source URL next to it.** That's what the cell below does, and it's what
you should do in your own codebase. When a price changes you edit one dict, not forty
call sites.

Verify against the source before you quote a number to anyone:
- **OpenAI** — <https://openai.com/api/pricing/>
- **Anthropic** — <https://www.anthropic.com/pricing#api>

In [ ]:
PRICING_AS_OF = "2026-06-24"

# Prices are USD per 1,000,000 tokens.
# ⚠️ Snapshot only — see the links above and update this dict when they change.
PRICING = {
    # ── Anthropic ───────────────────────────────────────────────
    "claude-opus-5":     {"in": 5.00,  "out": 25.00, "ctx": 1_000_000, "vendor": "anthropic"},
    "claude-sonnet-5":   {"in": 2.00,  "out": 10.00, "ctx": 1_000_000, "vendor": "anthropic"},
    "claude-haiku-4-5":  {"in": 1.00,  "out": 5.00,  "ctx":   200_000, "vendor": "anthropic"},
    # ── OpenAI ──────────────────────────────────────────────────
    "gpt-4.1":           {"in": 2.00,  "out": 8.00,  "ctx": 1_047_576, "vendor": "openai"},
    "gpt-4.1-mini":      {"in": 0.40,  "out": 1.60,  "ctx": 1_047_576, "vendor": "openai"},
    "gpt-4.1-nano":      {"in": 0.10,  "out": 0.40,  "ctx": 1_047_576, "vendor": "openai"},
    "gpt-4o":            {"in": 2.50,  "out": 10.00, "ctx":   128_000, "vendor": "openai"},
    "gpt-4o-mini":       {"in": 0.15,  "out": 0.60,  "ctx":   128_000, "vendor": "openai"},
}

# Prompt-caching multipliers, applied to the model's *input* price.
# Writing to the cache costs a little extra; reading from it is ~90% off.
CACHE_WRITE_MULTIPLIER = 1.25
CACHE_READ_MULTIPLIER = 0.10


def cost(model: str, input_tokens: int = 0, output_tokens: int = 0,
         cached_input_tokens: int = 0) -> float:
    """Dollar cost of a single request. `cached_input_tokens` are billed at the cache-read rate."""
    if model not in PRICING:
        raise KeyError(f"{model!r} is not in PRICING. Add it (and its source date).")
    p = PRICING[model]
    return (
        input_tokens / 1_000_000 * p["in"]
        + cached_input_tokens / 1_000_000 * p["in"] * CACHE_READ_MULTIPLIER
        + output_tokens / 1_000_000 * p["out"]
    )


def usd(amount: float) -> str:
    """Format a dollar amount that might be extremely small."""
    if amount == 0:
        return "$0"
    sign, m = ("-" if amount < 0 else ""), abs(amount)
    if m < 0.01:
        return f"{sign}${m:.6f}"
    if m < 1:
        return f"{sign}${m:.4f}"
    return f"{sign}${m:,.2f}"


print(f"Pricing snapshot: {PRICING_AS_OF} — {len(PRICING)} models loaded")
print(f"Sanity check: 1M in + 1M out on claude-opus-5 = {usd(cost('claude-opus-5', 1_000_000, 1_000_000))}")

### Demo 4.1 — What does one realistic request cost?

Let's price a request you'd actually make: a RAG-style call with a chunk of retrieved
context, asking for a short answer.

In [ ]:
INPUT_TOKENS = 2_000    # system prompt + retrieved context + user question
OUTPUT_TOKENS = 400     # a few paragraphs back

print(f"One request: {INPUT_TOKENS:,} input + {OUTPUT_TOKENS:,} output tokens\n")
print(f"{'model':<18} {'in $/M':>8} {'out $/M':>9} {'per call':>12} {'per 1k calls':>14} {'per 1M calls':>14}")
print("─" * 80)

priced = sorted(PRICING.items(), key=lambda kv: cost(kv[0], INPUT_TOKENS, OUTPUT_TOKENS))
for model, p in priced:
    c = cost(model, INPUT_TOKENS, OUTPUT_TOKENS)
    print(f"{model:<18} {p['in']:>8.2f} {p['out']:>9.2f} {usd(c):>12} "
          f"{usd(c * 1_000):>14} {usd(c * 1_000_000):>14}")

cheapest, priciest = priced[0][0], priced[-1][0]
ratio = cost(priciest, INPUT_TOKENS, OUTPUT_TOKENS) / cost(cheapest, INPUT_TOKENS, OUTPUT_TOKENS)
print(f"\n{priciest} costs {ratio:.0f}× more than {cheapest} for the identical request.")

### The "per 1M calls" column is the one to stare at

A single call costs a fraction of a cent. A million calls — a modest production feature —
is a real budget line, and the spread between the cheapest and priciest model is often
**30–50×**.

> 🎯 **This is why "just use the best model" is not a strategy, and neither is "just use
> the cheapest".** The right answer is: pick the cheapest model that passes your evals for
> *that specific task*. Classification and extraction rarely need a frontier model.
> Multi-step reasoning usually does. Module 6 teaches you how to measure this instead of
> guessing.

### Demo 4.2 — Output tokens are where the money goes

Same question. Two system prompts. One asks for brevity.

In [ ]:
MODEL = "claude-sonnet-5"

scenarios = {
    "Verbose (default behaviour)": {"input": 150, "output": 800},
    "Terse ('answer in 2 sentences')": {"input": 165, "output": 60},
}

print(f"Model: {MODEL}\n")
print(f"{'scenario':<34} {'in':>5} {'out':>5} {'input $':>10} {'output $':>10} {'total':>10}")
print("─" * 78)
for name, s in scenarios.items():
    ci = cost(MODEL, input_tokens=s["input"])
    co = cost(MODEL, output_tokens=s["output"])
    print(f"{name:<34} {s['input']:>5} {s['output']:>5} {usd(ci):>10} {usd(co):>10} {usd(ci + co):>10}")

verbose = cost(MODEL, **{"input_tokens": 150, "output_tokens": 800})
terse = cost(MODEL, **{"input_tokens": 165, "output_tokens": 60})
print(f"\nAdding 15 tokens of instruction ('be terse') made the call {verbose / terse:.1f}× cheaper.")
print(f"At 1M calls/month that is {usd(verbose * 1_000_000)} vs {usd(terse * 1_000_000)}.")

In [ ]:
models = list(PRICING.keys())
in_prices = [PRICING[m]["in"] for m in models]
out_prices = [PRICING[m]["out"] for m in models]
y = range(len(models))

fig, ax = plt.subplots(figsize=(9, 5))
ax.barh([i + 0.2 for i in y], in_prices, height=0.4, label="input  $/1M", color="#3b82f6")
ax.barh([i - 0.2 for i in y], out_prices, height=0.4, label="output $/1M", color="#ef4444")
ax.set_yticks(list(y), models)
ax.invert_yaxis()
ax.set_xlabel("USD per 1M tokens")
ax.set_title(f"Output always costs more than input  (snapshot {PRICING_AS_OF})",
             fontsize=13, weight="bold")
ax.legend()
ax.spines[["top", "right"]].set_visible(False)
for i, m in enumerate(models):
    ax.text(out_prices[i] + 0.15, i - 0.2, f"{out_prices[i]/in_prices[i]:.0f}×", va="center",
            fontsize=9, color="#ef4444")
plt.tight_layout()
plt.show()

### Three design rules that fall out of this chart

1. **Ask for less.** `"Answer in under 3 sentences."` is the highest-ROI sentence in your
   system prompt. So is a `max_tokens` ceiling.
2. **Prefer structured output over prose.** Getting `{"sentiment":"positive"}` back instead
   of a paragraph explaining the sentiment is a 20× output reduction for the same
   information.
3. **Stuffing context is comparatively cheap.** Don't contort your RAG pipeline to save 200
   input tokens while letting the model ramble for 800 output tokens. Optimise the
   expensive side first.

---
---

# Part 5 · Scaling — where the bill actually comes from

One request costs a fraction of a cent. Products don't make one request.

This part contains the mistake that catches almost every team building their first chat
feature. If you only take one thing from this notebook, take Demo 5.2.

### Demo 5.1 — Model your product's monthly bill

Before you write a line of code, you can answer "can we afford this?" Change the numbers
and re-run.

In [ ]:
def monthly_bill(model: str, daily_active_users: int, requests_per_user_per_day: int,
                 input_tokens: int, output_tokens: int, days: int = 30) -> dict:
    """Project a monthly LLM bill for a product."""
    per_call = cost(model, input_tokens, output_tokens)
    calls = daily_active_users * requests_per_user_per_day * days
    total = per_call * calls
    return {
        "model": model,
        "calls_per_month": calls,
        "cost_per_call": per_call,
        "monthly": total,
        "per_user_per_month": total / daily_active_users if daily_active_users else 0,
    }


PRODUCTS = {
    "Side project":      dict(daily_active_users=100,     requests_per_user_per_day=5,  input_tokens=1_500, output_tokens=300),
    "Seed startup":      dict(daily_active_users=5_000,   requests_per_user_per_day=12, input_tokens=2_500, output_tokens=500),
    "Series A SaaS":     dict(daily_active_users=50_000,  requests_per_user_per_day=20, input_tokens=4_000, output_tokens=600),
    "Consumer app":      dict(daily_active_users=500_000, requests_per_user_per_day=8,  input_tokens=1_200, output_tokens=250),
}

MODEL_CHOICES = ["gpt-4o-mini", "claude-haiku-4-5", "claude-sonnet-5", "claude-opus-5"]

print(f"{'product':<16} " + " ".join(f"{m:>17}" for m in MODEL_CHOICES))
print("─" * (16 + 18 * len(MODEL_CHOICES)))
for name, cfg in PRODUCTS.items():
    cells = [usd(monthly_bill(m, **cfg)["monthly"]) for m in MODEL_CHOICES]
    print(f"{name:<16} " + " ".join(f"{c:>17}" for c in cells))

print()
b = monthly_bill("claude-sonnet-5", **PRODUCTS["Seed startup"])
print(f"Seed startup on claude-sonnet-5:")
print(f"  {b['calls_per_month']:,} calls/month at {usd(b['cost_per_call'])} each")
print(f"  = {usd(b['monthly'])}/month, or {usd(b['per_user_per_month'])} per active user")
print()
print("👉 'Cost per active user per month' is the number your CFO cares about.")
print("   Compare it to your revenue per user. That comparison is your business model.")

### Demo 5.2 — ⚠️ The O(n²) trap: conversation history

**This is the big one.**

LLM APIs are **stateless**. The model remembers nothing between calls. To continue a
conversation, you resend the *entire history* every single turn.

So on turn 20, you're not paying for turn 20. You're paying for turns 1 through 20 — again.

Cumulative token spend grows **quadratically** with conversation length.

### The frontend analogy

It's the n+1 query problem, except you can't fix it with a `JOIN`, and every duplicate
costs real money.

In [ ]:
SYSTEM_TOKENS = 200      # your system prompt, resent every turn
USER_TOKENS = 60         # average user message
ASSISTANT_TOKENS = 250   # average model reply
TURNS = 40
SIM_MODEL = "claude-sonnet-5"


def simulate_naive(turns=TURNS, model=SIM_MODEL):
    """Resend the whole history every turn. The default if you don't think about it."""
    history_tokens = SYSTEM_TOKENS
    per_turn, cumulative = [], []
    running = 0.0
    for _ in range(1, turns + 1):
        history_tokens += USER_TOKENS                 # user adds their message
        input_tokens = history_tokens                 # ...and we resend all of it
        c = cost(model, input_tokens=input_tokens, output_tokens=ASSISTANT_TOKENS)
        history_tokens += ASSISTANT_TOKENS            # reply joins the history
        running += c
        per_turn.append(c)
        cumulative.append(running)
    return per_turn, cumulative


per_turn, cumulative = simulate_naive()

print(f"{'turn':>5} {'input tokens':>13} {'cost this turn':>16} {'cumulative':>13}")
print("─" * 52)
for t in (1, 5, 10, 20, 30, 40):
    i = t - 1
    input_tokens = SYSTEM_TOKENS + t * USER_TOKENS + (t - 1) * ASSISTANT_TOKENS
    print(f"{t:>5} {input_tokens:>13,} {usd(per_turn[i]):>16} {usd(cumulative[i]):>13}")

print()
print(f"Turn 40 costs {per_turn[-1] / per_turn[0]:.1f}× what turn 1 cost — for the same size message.")
print(f"Whole 40-turn conversation: {usd(cumulative[-1])}")
print(f"If it were linear (turn 1 price × 40): {usd(per_turn[0] * 40)}")
print(f"The quadratic term cost you {usd(cumulative[-1] - per_turn[0] * 40)} extra. Per conversation.")

### Demo 5.3 — Three ways to fight back, measured

Now the useful part: what do you actually *do* about it?

- **Naive** — resend everything. Perfect memory, quadratic cost.
- **Sliding window** — keep the system prompt + the last *k* turns. Cheap and trivial to
  implement; the model forgets anything older.
- **Summarise + window** — periodically compress old turns into a summary and keep recent
  turns verbatim. Keeps long-range memory, and we'll *include the cost of the summarising
  calls themselves*, because pretending they're free is how estimates lie.

In [ ]:
WINDOW_TURNS = 6          # how many recent turns to keep verbatim
SUMMARY_EVERY = 8         # re-summarise every N turns
SUMMARY_TOKENS = 150      # size of the running summary we carry
SUMMARIZER_MODEL = "claude-haiku-4-5"   # use a cheap model for compression!


def simulate_sliding(turns=TURNS, model=SIM_MODEL, window=WINDOW_TURNS):
    cumulative, running = [], 0.0
    for t in range(1, turns + 1):
        kept = min(t, window)
        # system + (kept-1) complete past turns + this turn's user message
        input_tokens = SYSTEM_TOKENS + (kept - 1) * (USER_TOKENS + ASSISTANT_TOKENS) + USER_TOKENS
        running += cost(model, input_tokens=input_tokens, output_tokens=ASSISTANT_TOKENS)
        cumulative.append(running)
    return cumulative


def simulate_summarised(turns=TURNS, model=SIM_MODEL, window=WINDOW_TURNS):
    cumulative, running = [], 0.0
    for t in range(1, turns + 1):
        kept = min(t, window)
        has_summary = t > window
        input_tokens = SYSTEM_TOKENS + (SUMMARY_TOKENS if has_summary else 0)
        input_tokens += (kept - 1) * (USER_TOKENS + ASSISTANT_TOKENS) + USER_TOKENS
        running += cost(model, input_tokens=input_tokens, output_tokens=ASSISTANT_TOKENS)

        # Every SUMMARY_EVERY turns we pay for an extra call to compress the old turns.
        if t > window and t % SUMMARY_EVERY == 0:
            to_compress = SUMMARY_EVERY * (USER_TOKENS + ASSISTANT_TOKENS) + SUMMARY_TOKENS
            running += cost(SUMMARIZER_MODEL,
                            input_tokens=to_compress, output_tokens=SUMMARY_TOKENS)
        cumulative.append(running)
    return cumulative


naive_c = cumulative
sliding_c = simulate_sliding()
summar_c = simulate_summarised()

print(f"{'turn':>5} {'naive':>12} {'sliding':>12} {'summarised':>12}")
print("─" * 44)
for t in (1, 10, 20, 30, 40):
    i = t - 1
    print(f"{t:>5} {usd(naive_c[i]):>12} {usd(sliding_c[i]):>12} {usd(summar_c[i]):>12}")

print()
print(f"After {TURNS} turns:")
print(f"  naive       {usd(naive_c[-1])}   (baseline)")
print(f"  sliding     {usd(sliding_c[-1])}   → {(1 - sliding_c[-1]/naive_c[-1])*100:.0f}% cheaper, forgets old turns")
print(f"  summarised  {usd(summar_c[-1])}   → {(1 - summar_c[-1]/naive_c[-1])*100:.0f}% cheaper, keeps the gist")

In [ ]:
turns_axis = list(range(1, TURNS + 1))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.8))

ax1.plot(turns_axis, [c * 1000 for c in naive_c], lw=2.4, color="#ef4444", label="naive (resend all)")
ax1.plot(turns_axis, [c * 1000 for c in sliding_c], lw=2.4, color="#3b82f6", label=f"sliding window ({WINDOW_TURNS} turns)")
ax1.plot(turns_axis, [c * 1000 for c in summar_c], lw=2.4, color="#22c55e", label="summarise + window")
ax1.set_xlabel("conversation turn")
ax1.set_ylabel("cumulative cost (US cents ×10)")
ax1.set_title("Cumulative cost of one conversation", weight="bold")
ax1.legend()
ax1.grid(alpha=0.25)
ax1.spines[["top", "right"]].set_visible(False)

ax2.plot(turns_axis, [c * 1000 for c in per_turn], lw=2.4, color="#ef4444")
ax2.fill_between(turns_axis, [c * 1000 for c in per_turn], alpha=0.15, color="#ef4444")
ax2.set_xlabel("conversation turn")
ax2.set_ylabel("cost of THIS turn (US cents ×10)")
ax2.set_title("Naive: every turn costs more than the last", weight="bold")
ax2.grid(alpha=0.25)
ax2.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.show()

### How to choose

| Strategy | Cost | Memory | Reach for it when |
|---|---|---|---|
| Naive | 💸💸💸 quadratic | perfect | conversations are genuinely short (< ~10 turns) |
| Sliding window | 💸 linear | recent only | support bots, Q&A, anything mostly turn-local |
| Summarise + window | 💸💸 near-linear | gist + recent | long assistant-style chats where early context matters |

Two things people get wrong:

1. **Summarise with a cheap model.** Compression is a mechanical task — `claude-haiku-4-5`
   or `gpt-4o-mini` do it fine. Using your frontier model to write summaries is pouring
   money away. (Notice `SUMMARIZER_MODEL` in the simulation above.)
2. **Count the summarising calls.** They're not free. The simulation includes them, which is
   why the green line isn't as far below the blue one as you'd expect.

> 💡 **There's a fourth option worth knowing about:** several providers now offer
> **server-side compaction** — the API summarises old context for you automatically. Great
> ergonomics, still not free. Same principle, someone else's implementation.

---
---

# Part 6 · Prompt caching — the 90% discount most people leave on the table

If a big chunk of your prompt is **identical across requests** — a long system prompt, a
few-shot example block, a document you're answering questions about — you can have the
provider cache it and charge you ~10% of the normal rate to re-read it.

### How the billing works

| | Multiplier on the input price |
|---|---|
| Normal input token | 1.00× |
| **Writing** to the cache (first time) | ~1.25× |
| **Reading** from the cache (subsequent) | ~0.10× |

### The one rule that decides whether it works

**Caching is a prefix match.** The provider hashes your prompt from the very beginning.
One byte different anywhere in the prefix and everything after it misses.

So the layout of your prompt matters:

```
┌──────────────────────────────────┐
│  STABLE — cache this             │   system prompt, few-shot examples,
│  (byte-identical every request)  │   tool definitions, the document
├──────────────────────────────────┤   ← cache breakpoint
│  VOLATILE — never cache          │   timestamps, user IDs, the question
└──────────────────────────────────┘
```

### The frontend analogy

Immutable, content-hashed asset filenames. `vendor.a3f9c1.js` gets cached forever because
its bytes never change. Bump one byte and it's a brand new URL and a fresh download. Your
prompt prefix is that file — **stop putting `Date.now()` in it.**

### Demo 6.1 — When does caching pay for itself?

Writing costs 1.25×, so a *single* cached request is slightly *more* expensive. Let's find
the break-even point.

In [ ]:
def cached_prefix_cost(model: str, prefix_tokens: int, n_requests: int) -> float:
    """Total input cost for n requests sharing one cached prefix."""
    p = PRICING[model]["in"]
    write = prefix_tokens / 1_000_000 * p * CACHE_WRITE_MULTIPLIER          # 1st request
    reads = prefix_tokens / 1_000_000 * p * CACHE_READ_MULTIPLIER * (n_requests - 1)
    return write + reads


def uncached_prefix_cost(model: str, prefix_tokens: int, n_requests: int) -> float:
    return prefix_tokens / 1_000_000 * PRICING[model]["in"] * n_requests


MODEL = "claude-sonnet-5"
PREFIX = 20_000     # e.g. a long system prompt + a document you keep asking about

print(f"Model {MODEL} · cached prefix of {PREFIX:,} tokens\n")
print(f"{'requests':>9} {'uncached':>12} {'cached':>12} {'saved':>12} {'saved %':>9}")
print("─" * 60)
for n in (1, 2, 3, 5, 10, 100, 1_000, 10_000):
    u = uncached_prefix_cost(MODEL, PREFIX, n)
    c = cached_prefix_cost(MODEL, PREFIX, n)
    print(f"{n:>9,} {usd(u):>12} {usd(c):>12} {usd(u - c):>12} {(u - c) / u * 100:>8.1f}%")

# Find the break-even point by just... checking. (Cheaper than algebra, harder to get wrong.)
breakeven = next(
    n for n in range(1, 1000)
    if cached_prefix_cost(MODEL, PREFIX, n) < uncached_prefix_cost(MODEL, PREFIX, n)
)
print(f"\nRequest 1 alone is {CACHE_WRITE_MULTIPLIER:.2f}× the normal price — caching costs "
      f"a little extra up front.")
print(f"Break-even at request {breakeven}. From there you converge on a flat "
      f"{(1 - CACHE_READ_MULTIPLIER) * 100:.0f}% discount.")

In [ ]:
ns = list(range(1, 51))
u_line = [uncached_prefix_cost(MODEL, PREFIX, n) for n in ns]
c_line = [cached_prefix_cost(MODEL, PREFIX, n) for n in ns]

fig, ax = plt.subplots(figsize=(9, 4.8))
ax.plot(ns, u_line, lw=2.4, color="#ef4444", label="no caching")
ax.plot(ns, c_line, lw=2.4, color="#22c55e", label="with prompt caching")
ax.fill_between(ns, c_line, u_line, alpha=0.12, color="#22c55e")
ax.set_xlabel("number of requests sharing the same prefix")
ax.set_ylabel("cumulative input cost (USD)")
ax.set_title(f"A {PREFIX:,}-token cached prefix on {MODEL}", weight="bold")
ax.annotate(f"saved {usd(u_line[-1] - c_line[-1])} by request 50",
            xy=(50, (u_line[-1] + c_line[-1]) / 2), xytext=(26, u_line[-1] * 0.55),
            arrowprops=dict(arrowstyle="->", color="#16a34a"), color="#16a34a", fontsize=10)
ax.legend()
ax.grid(alpha=0.25)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

> ⏱️ **The catch nobody mentions: cache entries expire.** The default TTL is around
> **5 minutes** of inactivity (longer TTLs are available at a higher write price). So the
> discount is real for *bursty, repeated* traffic — an active chat session, a batch job, a
> document Q&A loop. It does nothing for one lonely request an hour.

### Demo 6.2 — 💸 LIVE API: prove the cache is actually working

Estimated caching savings are worthless. `usage` tells you the truth. Two fields matter:

- `cache_creation_input_tokens` — tokens written to the cache (billed at ~1.25×)
- `cache_read_input_tokens` — tokens served from the cache (billed at ~0.10×)

We'll make the **same** request twice and watch the numbers move.

> 💰 **Cost of running this cell: roughly $0.01–0.02.** We deliberately use a cheap model
> and a ~4,000 token prefix (the minimum cacheable prefix is model-dependent — commonly
> 1,024–4,096 tokens — so a short system prompt silently won't cache at all).

In [ ]:
if not HAS_ANTHROPIC:
    print("⬜ Skipped — set ANTHROPIC_API_KEY in .env to run this cell.")
else:
    import anthropic

    aclient = anthropic.Anthropic()
    CACHE_DEMO_MODEL = "claude-haiku-4-5"

    # A stable, boringly long system prompt — this is what we want cached.
    STABLE_PREFIX = (
        "You are a documentation assistant for a fictional design system called Nimbus.\n\n"
        + "\n".join(
            f"- Component {i}: `<Nimbus{i} />` accepts props `size`, `tone`, and `onPress`. "
            f"It renders a focusable element and forwards refs. Prefer it over raw HTML "
            f"elements when you need consistent theming across the application surface."
            for i in range(1, 121)
        )
    )

    prefix_tokens = aclient.messages.count_tokens(
        model=CACHE_DEMO_MODEL,
        system=STABLE_PREFIX,
        messages=[{"role": "user", "content": "hi"}],
    ).input_tokens
    print(f"stable prefix ≈ {prefix_tokens:,} tokens\n")

    def ask(question: str):
        return aclient.messages.create(
            model=CACHE_DEMO_MODEL,
            max_tokens=80,
            system=[{
                "type": "text",
                "text": STABLE_PREFIX,
                "cache_control": {"type": "ephemeral"},   # ← the cache breakpoint
            }],
            messages=[{"role": "user", "content": question}],
        )

    def report(label, r):
        u = r.usage
        created = getattr(u, "cache_creation_input_tokens", 0) or 0
        read = getattr(u, "cache_read_input_tokens", 0) or 0
        billed = cost(
            CACHE_DEMO_MODEL,
            input_tokens=u.input_tokens + int(created * CACHE_WRITE_MULTIPLIER),
            output_tokens=u.output_tokens,
            cached_input_tokens=read,
        )
        print(f"{label:<16} uncached_in={u.input_tokens:<6} cache_write={created:<6} "
              f"cache_read={read:<6} out={u.output_tokens:<4} ≈ {usd(billed)}")

    report("call 1 (cold)", ask("What props does Nimbus7 accept?"))
    report("call 2 (warm)", ask("And what about Nimbus42?"))

    print("\n👉 Call 1 writes the cache. Call 2 reads it — same prefix, ~10% of the price.")
    print("   If cache_read is 0 on call 2, your prefix is below the model's minimum,")
    print("   or something in it changed between calls. Which brings us to...")

### Demo 6.3 — 💸 LIVE API: the silent cache killer

Here's a bug that costs real money and produces **no error, no warning, nothing in the
logs**. Your code looks correct. Your cache hit rate is just quietly zero.

The culprit: something dynamic in the cached prefix. A timestamp. A request ID. A
`random.shuffle()`. An unsorted `json.dumps()` of a dict.

> 💰 **Cost: roughly $0.01.**

In [ ]:
if not HAS_ANTHROPIC:
    print("⬜ Skipped — set ANTHROPIC_API_KEY in .env to run this cell.")
else:
    from datetime import datetime

    def ask_poisoned(question: str):
        """Looks harmless. The timestamp changes every call, so the prefix never matches."""
        poisoned_prefix = (
            f"Current time: {datetime.now().isoformat()}\n\n"   # 💀 the bug
            + STABLE_PREFIX
        )
        return aclient.messages.create(
            model=CACHE_DEMO_MODEL,
            max_tokens=80,
            system=[{"type": "text", "text": poisoned_prefix,
                     "cache_control": {"type": "ephemeral"}}],
            messages=[{"role": "user", "content": question}],
        )

    print("Same two calls, but with a timestamp at the TOP of the prefix:\n")
    report("call 1", ask_poisoned("What props does Nimbus7 accept?"))
    report("call 2", ask_poisoned("And what about Nimbus42?"))

    print("\n💀 cache_read = 0 on call 2. Every request pays full price, forever, silently.")
    print("   The fix is not to delete the timestamp — it's to MOVE it.")
    print("   Stable content first, volatile content after the cache breakpoint.")

### 🔍 The cache-miss audit checklist

If `cache_read_input_tokens` is 0 when you expected a hit, it's almost always one of these:

- [ ] A **timestamp, UUID, or request ID** inside the cached prefix
- [ ] `json.dumps(some_dict)` **without `sort_keys=True`** — Python dict order is stable,
      but the dict you built from a DB query or a `set` may not be
- [ ] Tool definitions that get **rebuilt in a different order** per request
- [ ] The prefix is **below the model's minimum** cacheable length (~1k–4k tokens)
- [ ] More than **5 minutes** since the last hit — the entry expired
- [ ] You changed **model, or temperature-adjacent settings** — caches are model-scoped
- [ ] Your "stable" system prompt interpolates the **user's name or locale**

> 🎯 **Ship this as a metric.** Log `cache_read_input_tokens / total_input_tokens` per
> request and alert when it drops. It's the cheapest cost regression alarm you'll ever build.

---
---

# Part 7 · Guardrails — the code you actually ship

Everything so far was analysis. This part is the thing you paste into your project.

Three layers, and you want all three:

1. **Preflight** — count tokens *before* sending; reject anything absurd
2. **Ceiling** — `max_tokens` caps the expensive side of every single call
3. **Accounting** — record real `usage` after every call; enforce a running budget

### The frontend analogy

It's the same shape as an error boundary plus a rate limiter plus your analytics. You don't
ship a payment form without validation. Don't ship an LLM call without a budget.

### Demo 7.1 — A `CostTracker` you can copy into your project

In [ ]:
from datetime import datetime, timezone


class BudgetExceeded(RuntimeError):
    """Raised when a call would push spend past the configured ceiling."""


@dataclass
class Call:
    ts: str
    model: str
    label: str
    input_tokens: int
    output_tokens: int
    cached_tokens: int
    cost: float


@dataclass
class CostTracker:
    """Records LLM spend and enforces a hard budget ceiling.

    Usage:
        tracker = CostTracker(budget_usd=5.00)
        tracker.preflight("claude-sonnet-5", input_tokens=2000, max_output_tokens=500)
        ... make the call ...
        tracker.record_openai(response, model="gpt-4o-mini", label="summarise")
    """

    budget_usd: float = 10.0
    warn_at: float = 0.8          # warn once we cross 80% of the budget
    calls: list = field(default_factory=list)
    _warned: bool = False

    # ── accounting ────────────────────────────────────────────
    @property
    def spent(self) -> float:
        return sum(c.cost for c in self.calls)

    @property
    def remaining(self) -> float:
        return max(0.0, self.budget_usd - self.spent)

    # ── layer 1: preflight ────────────────────────────────────
    def preflight(self, model: str, input_tokens: int, max_output_tokens: int,
                  label: str = "") -> float:
        """Estimate worst-case cost and refuse the call if it would blow the budget."""
        worst_case = cost(model, input_tokens, max_output_tokens)
        if self.spent + worst_case > self.budget_usd:
            raise BudgetExceeded(
                f"Call {label or model!r} could cost up to {usd(worst_case)}; "
                f"only {usd(self.remaining)} of the {usd(self.budget_usd)} budget remains."
            )
        return worst_case

    # ── layer 3: record what actually happened ────────────────
    def record(self, model: str, input_tokens: int, output_tokens: int,
               cached_tokens: int = 0, label: str = "") -> Call:
        c = Call(
            ts=datetime.now(timezone.utc).isoformat(timespec="seconds"),
            model=model, label=label,
            input_tokens=input_tokens, output_tokens=output_tokens,
            cached_tokens=cached_tokens,
            cost=cost(model, input_tokens, output_tokens, cached_tokens),
        )
        self.calls.append(c)
        if not self._warned and self.spent >= self.budget_usd * self.warn_at:
            self._warned = True
            print(f"⚠️  Budget warning: {usd(self.spent)} of {usd(self.budget_usd)} used "
                  f"({self.spent / self.budget_usd * 100:.0f}%).")
        return c

    # ── provider adapters: read `usage` off a real response ───
    def record_openai(self, response, model: str, label: str = "") -> Call:
        u = response.usage
        cached = getattr(getattr(u, "prompt_tokens_details", None), "cached_tokens", 0) or 0
        return self.record(model, u.prompt_tokens - cached, u.completion_tokens, cached, label)

    def record_anthropic(self, response, model: str, label: str = "") -> Call:
        u = response.usage
        created = getattr(u, "cache_creation_input_tokens", 0) or 0
        read = getattr(u, "cache_read_input_tokens", 0) or 0
        # cache writes bill at ~1.25x input, so charge them as inflated normal input
        return self.record(model, u.input_tokens + int(created * CACHE_WRITE_MULTIPLIER),
                           u.output_tokens, read, label)

    # ── reporting ─────────────────────────────────────────────
    def report(self) -> None:
        if not self.calls:
            print("No calls recorded.")
            return
        by_label: dict[str, list] = {}
        for c in self.calls:
            by_label.setdefault(c.label or c.model, []).append(c)

        print(f"{'label':<22} {'calls':>6} {'in tk':>9} {'out tk':>8} {'cached':>8} {'cost':>11}")
        print("─" * 68)
        for label, cs in sorted(by_label.items(), key=lambda kv: -sum(c.cost for c in kv[1])):
            print(f"{label:<22} {len(cs):>6} {sum(c.input_tokens for c in cs):>9,} "
                  f"{sum(c.output_tokens for c in cs):>8,} {sum(c.cached_tokens for c in cs):>8,} "
                  f"{usd(sum(c.cost for c in cs)):>11}")
        print("─" * 68)
        pct = self.spent / self.budget_usd * 100
        bar = "█" * int(pct / 5) + "░" * (20 - int(pct / 5))
        print(f"{'TOTAL':<22} {len(self.calls):>6} {'':>9} {'':>8} {'':>8} {usd(self.spent):>11}")
        print(f"\nbudget  [{bar}] {pct:.1f}%   {usd(self.spent)} / {usd(self.budget_usd)}   "
              f"({usd(self.remaining)} left)")

    def to_dataframe(self):
        import pandas as pd
        return pd.DataFrame([c.__dict__ for c in self.calls])


print("CostTracker ready.")

### Demo 7.2 — Run a simulated workload through it

No API keys needed — we feed it realistic usage numbers so you can see the guardrail fire.

In [ ]:
import random

random.seed(42)
tracker = CostTracker(budget_usd=0.50)

WORKLOAD = [
    ("classify-intent",  "claude-haiku-4-5",  (300, 600),   (10, 30)),
    ("rag-answer",       "claude-sonnet-5",   (2500, 4500), (200, 600)),
    ("summarise-thread", "claude-haiku-4-5",  (1200, 3000), (80, 200)),
    ("final-review",     "claude-opus-5",     (3000, 5000), (400, 900)),
]

for i in range(60):
    label, model, in_range, out_range = random.choice(WORKLOAD)
    input_tokens = random.randint(*in_range)
    output_tokens = random.randint(*out_range)
    try:
        tracker.preflight(model, input_tokens, max_output_tokens=out_range[1], label=label)
    except BudgetExceeded as e:
        print(f"\n🛑 Request #{i + 1} blocked: {e}")
        break
    tracker.record(model, input_tokens, output_tokens, label=label)

print()
tracker.report()

In [ ]:
df = tracker.to_dataframe()

by_label = df.groupby("label")["cost"].agg(["sum", "count", "mean"]).sort_values("sum", ascending=False)
by_label.columns = ["total_$", "calls", "avg_$_per_call"]
print(by_label.to_string(float_format=lambda v: f"{v:.6f}"))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.4))

ax1.pie(by_label["total_$"], labels=by_label.index, autopct="%1.0f%%", startangle=140,
        colors=["#ef4444", "#f59e0b", "#3b82f6", "#22c55e"])
ax1.set_title("Where the money went", weight="bold")

ax2.plot(range(1, len(df) + 1), df["cost"].cumsum(), lw=2.4, color="#3b82f6")
ax2.axhline(tracker.budget_usd, ls="--", color="#ef4444", lw=1.5, label="budget ceiling")
ax2.set_xlabel("request #")
ax2.set_ylabel("cumulative spend (USD)")
ax2.set_title("Spend against the ceiling", weight="bold")
ax2.legend()
ax2.grid(alpha=0.25)
ax2.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.show()

# This is the artefact you'd ship to your data warehouse:
print("\nfirst 3 rows of the audit log:")
print(df.head(3).to_string(index=False))

### Demo 7.3 — 💸 LIVE API: the tracker against a real call

The same tracker, but `record_openai` / `record_anthropic` pull the *real* numbers off the
response instead of us inventing them.

> 💰 **Cost: a fraction of a cent.**

In [ ]:
live = CostTracker(budget_usd=1.00)

if HAS_OPENAI:
    from openai import OpenAI
    oclient = OpenAI()
    msgs = [{"role": "user", "content": "In one sentence: what is a token in an LLM?"}]
    live.preflight("gpt-4o-mini", num_tokens_from_messages(msgs), 80, label="explain-token")
    r = oclient.chat.completions.create(model="gpt-4o-mini", messages=msgs, max_tokens=80)
    live.record_openai(r, model="gpt-4o-mini", label="explain-token")
    print("openai   :", r.choices[0].message.content.strip())

if HAS_ANTHROPIC:
    import anthropic
    aclient = anthropic.Anthropic()
    r = aclient.messages.create(
        model="claude-haiku-4-5",
        max_tokens=80,
        messages=[{"role": "user", "content": "In one sentence: what is a token in an LLM?"}],
    )
    live.record_anthropic(r, model="claude-haiku-4-5", label="explain-token")
    print("anthropic:", r.content[0].text.strip())

print()
if live.calls:
    live.report()
else:
    print("⬜ Skipped — no API keys set.")

### 📋 Production checklist

Steal this list.

- [ ] **`max_tokens` on every call.** It is a hard ceiling on your most expensive dimension.
      Set it deliberately; don't let a runaway generation write you a novel.
- [ ] **Log `usage` on every response** — input, output, cached, model, latency, and a
      `label` for the feature that made the call. Without the label you can't answer
      "which feature is expensive?"
- [ ] **Budget per user, per tenant, and globally.** All three. A single abusive account
      shouldn't be able to spend your monthly budget on a Tuesday.
- [ ] **Alert on cost-per-request, not just total cost.** Total cost rising because you have
      more users is good news. Cost *per request* rising is a bug.
- [ ] **Track cache hit rate.** A silent drop to zero is a real and common incident.
- [ ] **Set spend limits in the provider console too.** Your code is not the last line of
      defence — the provider's hard cap is.

---
---

# Part 8 · Context windows and the "1M context" trap

The **context window** is the maximum tokens a model can hold at once — input *plus* the
output it generates. Modern models advertise enormous ones (200K, 1M+).

Two things people get wrong about that number.

**1. Output shares the window.** If you're 500 tokens from the limit, you get 500 tokens of
answer, and then it truncates mid-sentence.

**2. Big window ≠ you should fill it.** The window is a *capacity* limit. Your bill is a
*usage* charge. Filling a 1M-token window is technically allowed and financially
spectacular.

In [ ]:
print(f"{'model':<18} {'context':>12} {'cost to fill it ONCE':>22} {'…× 1,000 requests':>20}")
print("─" * 76)
for model, p in sorted(PRICING.items(), key=lambda kv: -kv[1]["ctx"]):
    full = cost(model, input_tokens=p["ctx"])
    print(f"{model:<18} {p['ctx']:>12,} {usd(full):>22} {usd(full * 1_000):>20}")

print("\n👉 A 1M-token context window on a frontier model is a ~$5 button.")
print("   Press it a thousand times a day and it's a $5,000/day habit.")
print("   Retrieval exists precisely so you send 4,000 relevant tokens instead of 1,000,000.")

### A helper worth having around

In [ ]:
def fits(model: str, input_tokens: int, max_output_tokens: int) -> dict:
    """Will this request fit, and how much headroom is left?"""
    limit = PRICING[model]["ctx"]
    needed = input_tokens + max_output_tokens
    return {
        "fits": needed <= limit,
        "needed": needed,
        "limit": limit,
        "headroom": limit - needed,
        "used_pct": needed / limit * 100,
    }


checks = [
    ("claude-haiku-4-5", 190_000, 20_000),   # over
    ("claude-haiku-4-5", 150_000, 4_000),    # tight but fine
    ("gpt-4o-mini",      126_000, 4_000),    # over by a hair — the worst kind
    ("claude-sonnet-5",  400_000, 8_000),    # roomy
]

for model, i, o in checks:
    r = fits(model, i, o)
    icon = "✅" if r["fits"] else "❌"
    print(f"{icon} {model:<18} {i:>8,} in + {o:>7,} out = {r['needed']:>9,} / "
          f"{r['limit']:>9,}  ({r['used_pct']:>5.1f}% used, "
          f"{r['headroom']:>+9,} headroom)  {usd(cost(model, i, o))}")

> 📎 We go much deeper on windows, truncation strategy, and "lost in the middle"
> degradation in **`04_context_window_limits.ipynb`**. For now, the cost lesson is enough:
> **a big context window is a capability, not a plan.**

---
---

# Part 9 · Your turn

These are the exercises that make the material stick. Do them before the next episode —
each one is a small, real thing you'd actually build.

### 🏋️ Exercise 1 — Token-count your own codebase

Point this at a real project directory and find out what it would cost to hand your codebase
to a model. Then answer: which file type is your most expensive per byte?

In [ ]:
from pathlib import Path


def count_repo_tokens(root: str, extensions=(".py", ".ts", ".tsx", ".js", ".jsx", ".md"),
                      skip_dirs=(".git", "node_modules", ".venv", "dist", "build", "__pycache__")):
    """TODO: walk `root`, tokenise each matching file, and return per-extension totals.

    Return something like:
        {".ts": {"files": 42, "chars": 120_000, "tokens": 38_000}, ...}

    Then print a table sorted by tokens, plus the total cost of sending the whole
    thing to claude-sonnet-5 as input.

    Hints:
      - Path(root).rglob("*") to walk
      - `if any(part in skip_dirs for part in path.parts): continue`
      - read with encoding="utf-8", errors="ignore" so a stray binary doesn't crash you
    """
    raise NotImplementedError("Your turn!")


# count_repo_tokens("..")   # ← the llm-101 repo itself is a fine place to start

### 🏋️ Exercise 2 — Build `pick_cheapest_model()`

Given a task's token profile and a list of models that are "good enough" for it, return the
cheapest. This function is the seed of a real cost-routing layer.

In [ ]:
def pick_cheapest_model(input_tokens: int, output_tokens: int,
                        candidates: list[str], max_context: int | None = None) -> str:
    """TODO: return the cheapest candidate that (a) fits the context and
    (b) has a price entry. Raise a clear error if nothing qualifies.

    Bonus: add a `quality_floor` argument and a QUALITY dict so you can express
    "must be at least as capable as claude-haiku-4-5" — that's what real routing does.
    """
    raise NotImplementedError("Your turn!")


# assert pick_cheapest_model(2000, 400, ["claude-opus-5", "gpt-4o-mini", "claude-sonnet-5"]) == "gpt-4o-mini"

### 🏋️ Exercise 3 — Find the cheapest prompt format for *your* data

Take a real API response from a project you've worked on. Encode it four ways (pretty JSON,
minified JSON, CSV, Markdown table) and measure the token difference.

Then the part everyone skips: **send all four to a model with the same question and check
whether the answers are equally good.** Cheapest-that-still-works is the target, not
cheapest.

In [ ]:
MY_DATA = [
    # TODO: paste a real payload here — a list of dicts from an API you've built
]

# TODO: reuse as_pretty_json / as_compact_json / as_csv / as_markdown_table from Part 2,
#       print a token + cost comparison, then eyeball the model's answer quality for each.

### 🏋️ Exercise 4 — Instrument the O(n²) trap in something real

Take the `CostTracker` from Part 7 and wire it into a 20-turn conversation loop that resends
the full history. Watch `cost_per_call` climb in the log.

Then implement a sliding window and prove — with numbers from the tracker, not from the
simulation — how much you saved.

> This one is the strongest thing you can put in a portfolio from Module 0. "I found and
> fixed a quadratic cost bug, here's the before/after chart" is a genuinely good interview
> answer.

---
---

# 🎓 The cheat sheet

Screenshot this one.

### Counting

| Provider | Method | Notes |
|---|---|---|
| OpenAI | `tiktoken.encoding_for_model(m).encode(text)` | local, free, instant |
| Anthropic | `client.messages.count_tokens(...)` | free API call, exact |
| Anyone | `response.usage` | **the billed truth** — always log it |

### The numbers to remember

- **~4 chars per token** for English prose — **and nothing else**
- **Output costs 4–5× input** — optimise the output side first
- **Cache read ≈ 10%** of the input price; cache write ≈ 125%; break-even at **2 requests**
- **Non-English text costs 2–4× more** than the equivalent English
- **UUIDs and base64 are ~1 token per 1–2 chars** — never paste them in casually

### The four mistakes that cost real money

1. 🔁 **Resending full conversation history** → quadratic growth. Window or summarise.
2. 💬 **Not capping output** → set `max_tokens` *and* say "be brief" in the system prompt.
3. 🕐 **A timestamp in your cached prefix** → silent 100% cache miss, no error, no log line.
4. 🎯 **Frontier model for a trivial task** → route classification and extraction to a small
   model. Measure with evals, don't guess.

### The three-layer guardrail

```
preflight (count before sending)  →  max_tokens (cap the call)  →  record usage (enforce budget)
```

---

## ✅ Before you move on

- [ ] You can explain why `"hello"` and `" hello"` are different tokens
- [ ] You know which tokenizer to use for which provider, and why it matters
- [ ] You can price any request in dollars from its token counts
- [ ] You can explain the O(n²) history trap to a colleague at a whiteboard
- [ ] You have a `CostTracker` in your own project

## ➡️ Next up

**`02_transformer_intuition.ipynb`** — now that you know the model reads integers, we look at
what it *does* with them. Attention, visually, no heavy maths.

---

*Found a bug or a stale price? Open an issue — this notebook is meant to stay current.*